# CV-Pilot edge cases: what happens when things go wrong

The first notebook showed the happy path. This one is about the unhappy one: damaged inputs, a regression that hides
in an average, a file edited by hand, a training job killed twice. Every case follows the same pattern:

1. a headline, **What could go wrong**,
2. the setup (real data, real model outputs, then something deliberately broken),
3. the real command and its output **with the exit code** (0 = fine, 1 = a check failed or something is not
   trustworthy, 2 = the command was used wrongly),
4. **What this shows** in plain English.

The last cell collects every case in a table of *expected* against *actual* result. The expected values were
written down **before** each command ran, from the CV-Pilot documentation, and the table is built from the real outputs;
a row where they differ would be reported, never edited away.

The data and the model are the same as in the first notebook: the 128 real COCO128 photos (COCO annotations are
CC BY 4.0, the images keep their own Flickr licences, nothing is redistributed) and the real YOLO11n detector
(the `ultralytics` package is AGPL-3.0 and is the optional extra `cvpilot[ultralytics]`). If the first notebook
already ran, its detections are reused from the cache folder. Every command exits non-zero on purpose in many
cases; the exit code is printed instead of letting the cell fail.

### Getting ready

The same setup as in the first notebook: a fresh scratch folder, the small `sh()` helper, the downloads, the detector and the real labels.

In [1]:
import contextlib
import hashlib
import json
import os
import re
import shlex
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings("ignore")

# Where downloads are cached. This is outside the repository and nothing in it is ever committed.
DATA = Path(
    os.environ.get("CVPILOT_NOTEBOOK_DATA") or Path(tempfile.gettempdir()) / "cvpilot-notebooks"
)
DATA.mkdir(parents=True, exist_ok=True)
# A fresh scratch folder for this run of the notebook: everything the notebook writes lives here.
WORK = Path(tempfile.mkdtemp(prefix="cvpilot-nb-"))
os.chdir(WORK)

os.environ["CVPILOT_STORE_DIR"] = "store"  # runs, metrics and evaluation results
os.environ["CVPILOT_CHECKPOINT_DIR"] = "ckpt"  # checkpoints
os.environ["CVPILOT_PROJECT"] = "coco128-yolo"  # default project of every cvpilot command
os.environ["CVPILOT_LOG_LEVEL"] = "WARNING"  # keeps this notebook's own Python calls quiet
os.environ["YOLO_VERBOSE"] = "False"  # silence the progress bars of Ultralytics
os.environ["YOLO_CONFIG_DIR"] = str(DATA / "ultralytics-config")  # keep Ultralytics settings here
(DATA / "ultralytics-config").mkdir(exist_ok=True)
TIMINGS = {}  # seconds per step, printed at the end

print("python", sys.version.split()[0], "| scratch folder: a fresh temporary directory")
print("downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks)")

python 3.13.9 | scratch folder: a fresh temporary directory
downloads are cached in $CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks)


In [2]:
_REPLACEMENTS = sorted(
    [
        (str(WORK.resolve()), "."),
        (str(WORK), "."),
        (str(DATA.resolve()), "$CVPILOT_NOTEBOOK_DATA"),
        (str(DATA), "$CVPILOT_NOTEBOOK_DATA"),
        (str(Path.home()), "~"),
        (socket.gethostname(), "<this-computer>"),
    ],
    key=lambda pair: -len(pair[0]),
)
_OTHER_PATHS = re.compile(r"/(?:Users|home|private|var|tmp|opt)/[^\s\"',:)]+")
_QUIET_FIELDS = {"logger", "request_id", "service", "env", "@timestamp", "level", "message"}
_CLI = "import sys; sys.argv[0] = 'cvpilot'; from cvpilot.cli.app import main; main()"


def scrub(text):
    """Hide the paths and the computer name of the machine that runs this notebook."""
    for old, new in _REPLACEMENTS:
        if old:
            text = text.replace(old, new)
    return _OTHER_PATHS.sub("<path>", text)


def log_summary(line):
    """cvpilot writes one JSON object per log line on stderr; show it as one short line."""
    try:
        event = json.loads(line)
    except ValueError:
        return "stderr: " + scrub(line)
    if not isinstance(event, dict):
        return "stderr: " + scrub(line)
    fields = " ".join(
        f"{key}={scrub(str(value))[:44]}"
        for key, value in event.items()
        if key not in _QUIET_FIELDS
    )
    return f"log[{event.get('level', '?')}] {event.get('message')} {fields}"


class Result:
    def __init__(self, code, out, err):
        self.code, self.out, self.err = code, out, err

    def json(self):
        return json.loads(self.out)


def sh(
    command, *, lines=40, log_lines=8, quiet=False, keep=False, env=None, expect=None, timeout=900
):
    """Run one command, print what it printed (paths hidden) and its exit code.

    `cvpilot ...` runs the command-line tool of this Python environment. Output that is not
    JSON on stderr is shown as it is; the JSON log lines are shortened to one line each.
    The result (exit code, output) is returned only with `keep=True` or `quiet=True`.
    """
    argv = shlex.split(command)
    if argv[0] == "cvpilot":
        argv = [sys.executable, "-c", _CLI, *argv[1:]]
    elif argv[0] == "python":
        argv[0] = sys.executable
    full_env = {**os.environ, "CVPILOT_LOG_LEVEL": "INFO", **(env or {})}
    proc = subprocess.run(  # noqa: S603
        argv, capture_output=True, text=True, cwd=WORK, env=full_env, timeout=timeout, check=False
    )
    result = Result(proc.returncode, scrub(proc.stdout), scrub(proc.stderr))
    if not quiet:
        print("$", scrub(command))
        out_lines = result.out.rstrip("\n").splitlines()
        for line in out_lines[:lines]:
            print(line[:180] + (" ..." if len(line) > 180 else ""))
        if len(out_lines) > lines:
            print(f"... ({len(out_lines) - lines} more lines)")
        err_lines = [ln for ln in proc.stderr.splitlines() if ln.strip()]
        for line in err_lines[:log_lines]:
            print(log_summary(line)[:200])
        if len(err_lines) > log_lines:
            print(f"... ({len(err_lines) - log_lines} more log lines)")
        print("exit code:", proc.returncode)
    if expect is not None and proc.returncode != expect:
        raise RuntimeError(f"expected exit code {expect}, got {proc.returncode}: {command}")
    return result if (keep or quiet) else None


@contextlib.contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    TIMINGS[label] = round(time.perf_counter() - start, 1)
    print(f"[{label}: {TIMINGS[label]} s]")

### Checking what we download

Both downloads come from GitHub release assets of `ultralytics/assets` over `https`, with a size limit and a timeout, into a temporary file that is renamed only after the check.

* **Pinned fingerprints.** The expected SHA-256 of each file is written into the next cell. A file with a different digest is deleted and the notebook stops, before anything is unpacked or loaded; a cached file that does not match is never reused. GitHub publishes no digest for these two assets (the release API returns `digest: null`), so the pins are **trust on first use**, cross-checked in two ways: the file sizes equal the sizes the release API reports (6,983,030 and 5,613,764 bytes), and downloading both files a second time gave identical digests. To update a pin after a deliberate upstream change, download the new file, review it, and replace the constant.
* **Safe unpacking.** The zip is never extracted with `extractall`. Every member is checked first (no absolute path, no `..`, no symlink or special file, at most 1000 files and 50 MB in total, compression ratio at most 20) and written into a fresh folder.
* **A `.pt` file is a pickle.** Loading a PyTorch `.pt` file can run arbitrary code, so the YOLO11n weights are loaded only after the digest check passed. The checkpoints that *we* write (the classifier in this notebook) use CV-Pilot's `safetensors` format, which contains only numbers and can never run code.

In [3]:
COCO128_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
WEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt"
# Pinned SHA-256 of every download. A file with any other digest is deleted and the notebook stops.
PINNED_SHA256 = {
    "coco128.zip": "61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e",
    "yolo11n.pt": "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
}
MAX_DOWNLOAD_BYTES = {"coco128.zip": 20_000_000, "yolo11n.pt": 20_000_000}
ZIP_LIMITS = {"files": 1000, "bytes": 50_000_000, "ratio": 20}


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(url, name):
    """Download over https into the cache (bounded, via a temporary file) and check the SHA-256.

    A cached file with the wrong digest is deleted and fetched again; a download with the wrong
    digest is deleted and raises. Nothing is unpacked or loaded before this check passes."""
    dest = DATA / name
    if dest.exists() and sha256_of(dest) != PINNED_SHA256[name]:
        dest.unlink()  # never reuse a cached file that does not match
    if not dest.exists():
        if not url.startswith("https://"):
            raise ValueError("only https downloads are allowed")
        part = dest.with_name(name + ".part")
        try:
            with urllib.request.urlopen(url, timeout=60) as response, open(part, "wb") as out:  # noqa: S310
                total = 0
                while chunk := response.read(1 << 20):
                    total += len(chunk)
                    if total > MAX_DOWNLOAD_BYTES[name]:
                        raise RuntimeError(
                            f"{name} is larger than {MAX_DOWNLOAD_BYTES[name]} bytes"
                        )
                    out.write(chunk)
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            part.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not download {name} ({type(exc).__name__}). This notebook needs the "
                f"internet once. Download {url} yourself, put the file in the folder named by "
                "CVPILOT_NOTEBOOK_DATA (default: <tmp>/cvpilot-notebooks) and run this cell again."
            ) from None
        except RuntimeError:
            part.unlink(missing_ok=True)
            raise
        digest = sha256_of(part)
        if digest != PINNED_SHA256[name]:
            part.unlink()
            raise RuntimeError(f"{name}: SHA-256 {digest} is not the pinned {PINNED_SHA256[name]}")
        part.replace(dest)  # atomic: a half-downloaded file never has the final name
    print(
        f"{name:12s} {dest.stat().st_size / 1e6:5.1f} MB  sha256 {sha256_of(dest)}  (matches the pin)"
    )
    return dest


def safe_unzip(zip_path, target):
    """Unpack into a new folder after checking every member; never `extractall`."""
    if target.exists():
        return
    staging = Path(tempfile.mkdtemp(dir=DATA, prefix="unzip-"))
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        too_big = sum(m.file_size for m in members) > ZIP_LIMITS["bytes"]
        if len(members) > ZIP_LIMITS["files"] or too_big:
            raise RuntimeError("the archive has too many files or is too large")
        for member in members:
            name = Path(member.filename)
            mode = member.external_attr >> 16
            if name.is_absolute() or ".." in name.parts or "\\" in member.filename:
                raise RuntimeError(f"unsafe name in the archive: {member.filename}")
            if mode and (mode & 0o170000) not in (0o100000, 0o040000):
                raise RuntimeError(f"not a plain file or folder: {member.filename}")
            ratio = member.file_size / max(member.compress_size, 1)
            if ratio > ZIP_LIMITS["ratio"]:
                raise RuntimeError(f"suspicious compression ratio: {member.filename}")
        for member in members:
            out = staging / member.filename
            if member.is_dir():
                out.mkdir(parents=True, exist_ok=True)
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)
    staging.rename(target)


coco_zip = fetch(COCO128_URL, "coco128.zip")
weights_path = fetch(WEIGHTS_URL, "yolo11n.pt")  # a pickle: only loaded after the check above
COCO_DIR = DATA / "coco128-unpacked"
safe_unzip(coco_zip, COCO_DIR)
COCO_ROOT = COCO_DIR / "coco128"
IMAGES = sorted((COCO_ROOT / "images" / "train2017").glob("*.jpg"))
print(f"{len(IMAGES)} images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017")

coco128.zip    7.0 MB  sha256 61e5e3028863d8ffc3b81d6a514603954889f0edd5e4b44c4ce60b2da99aeb8e  (matches the pin)
yolo11n.pt     5.6 MB  sha256 0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1  (matches the pin)
128 images in $CVPILOT_NOTEBOOK_DATA/coco128-unpacked/coco128/images/train2017


In [4]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image

with timed("import torch and ultralytics, load YOLO11n"):
    import torch
    import ultralytics
    from ultralytics import YOLO

    ultralytics.settings.update({"sync": False})  # no usage reporting from this notebook
    MODEL = YOLO(str(weights_path))
NAMES = [MODEL.names[i] for i in range(len(MODEL.names))]
Path("classes.txt").write_text("\n".join(NAMES) + "\n")  # one class name per line, in id order
print(f"torch {torch.__version__}, ultralytics {ultralytics.__version__}, {len(NAMES)} classes")
print("device used for the detector: cpu")

[import torch and ultralytics, load YOLO11n: 0.8 s]
torch 2.14.1, ultralytics 8.4.177, 80 classes
device used for the detector: cpu


In [5]:
def yolo_ground_truth(root):
    """COCO128 labels are YOLO text files: `class x_center y_center width height`, all as
    fractions of the image. CV-Pilot wants pixel corners, one row per box."""
    cols = {k: [] for k in ("image_id", "cls", "x1", "y1", "x2", "y2", "width", "height")}
    for image in IMAGES:
        with Image.open(image) as handle:
            width, height = handle.size
        label = root / "labels" / "train2017" / f"{image.stem}.txt"
        lines = (
            label.read_text().splitlines() if label.exists() else []
        )  # a missing file: no objects
        boxes = [ln.split() for ln in lines if ln.strip()]
        if not boxes:  # an image with no objects: one row whose class and box are empty
            boxes = [[None] * 5]
        for cls, xc, yc, bw, bh in boxes:
            row = dict(image_id=image.stem, width=width, height=height)
            if cls is None:
                row.update(cls=None, x1=None, y1=None, x2=None, y2=None)
            else:
                xc, yc, bw, bh = (
                    float(xc) * width,
                    float(yc) * height,
                    float(bw) * width,
                    float(bh) * height,
                )
                row.update(
                    cls=int(cls), x1=xc - bw / 2, y1=yc - bh / 2, x2=xc + bw / 2, y2=yc + bh / 2
                )
            for key, value in row.items():
                cols[key].append(value)
    schema = pa.schema(
        [("image_id", pa.string()), ("cls", pa.int64())]
        + [(k, pa.float64()) for k in ("x1", "y1", "x2", "y2")]
        + [("width", pa.int64()), ("height", pa.int64())]
    )
    return pa.table(cols, schema=schema)


def show_rows(table, n=4):
    """Print the first rows of a Parquet table as plain text."""
    names = table.column_names
    print(" ".join(f"{name:>12}" for name in names))
    for row in table.slice(0, n).to_pylist():
        cells = [f"{v:.1f}" if isinstance(v, float) else str(v) for v in row.values()]
        print(" ".join(f"{cell:>12}" for cell in cells))


def detect(imgsz, *, use_cache=False):
    """Run YOLO11n on the 128 images and return detections as a table (also cached on disk)."""
    cache = (
        DATA / "predictions" / f"yolo11n-{PINNED_SHA256['yolo11n.pt'][:8]}-imgsz{imgsz}-cpu.parquet"
    )
    if use_cache and cache.exists():
        return pq.read_table(cache), 0.0, True
    start = time.perf_counter()
    cols = {k: [] for k in ("image_id", "cls", "score", "x1", "y1", "x2", "y2")}
    for lo in range(0, len(IMAGES), 16):
        batch = IMAGES[lo : lo + 16]
        # conf=0.001 keeps even very unsure boxes: mAP needs the whole precision/recall curve.
        results = MODEL.predict(
            [str(p) for p in batch],
            imgsz=imgsz,
            conf=0.001,
            iou=0.7,
            max_det=300,
            device="cpu",
            verbose=False,
        )
        for image, result in zip(batch, results, strict=True):
            boxes = result.boxes
            for (x1, y1, x2, y2), score, cls in zip(
                boxes.xyxy.numpy(), boxes.conf.numpy(), boxes.cls.numpy(), strict=True
            ):
                for key, value in zip(
                    cols,
                    (image.stem, int(cls), float(score), *map(float, (x1, y1, x2, y2))),
                    strict=True,
                ):
                    cols[key].append(value)
    table = pa.table(
        {
            "image_id": pa.array(cols["image_id"], pa.string()),
            "cls": pa.array(cols["cls"], pa.int64()),
            **{k: pa.array(cols[k], pa.float64()) for k in ("score", "x1", "y1", "x2", "y2")},
        }
    )
    cache.parent.mkdir(exist_ok=True)
    pq.write_table(table, str(cache) + ".part")
    Path(str(cache) + ".part").replace(cache)
    return table, time.perf_counter() - start, False

**What this shows.** Nothing new here: see the first notebook. `sh()` runs a command and prints its output and exit code; paths of this computer are replaced by placeholders.

In [6]:
import pyarrow.compute as pc

import cvpilot

gt = yolo_ground_truth(COCO_ROOT)
pq.write_table(gt, "gt.parquet")
preds640, secs640, cached640 = detect(640, use_cache=True)
preds480, secs480, cached480 = detect(480, use_cache=True)
print(
    "detections at 640 px:",
    len(preds640),
    "(from the cache)" if cached640 else f"(computed in {secs640:.1f} s)",
)
print(
    "detections at 480 px:",
    len(preds480),
    "(from the cache)" if cached480 else f"(computed in {secs480:.1f} s)",
)
for name, table in (("preds_640.parquet", preds640), ("preds_480.parquet", preds480)):
    pq.write_table(table, name)

CASES = []  # one row per case: what we expected, what happened


def new_run(name, project=None, config=None):
    """An empty, finished run to attach an evaluation to."""
    run = cvpilot.init(project, task="detection", name=name, config=config or {"name": name})
    cvpilot.finish()
    return run.run_id


def evaluate(name, preds, gts="gt.parquet", extra="", show=True, lines=12, **kw):
    """Create a run and evaluate predictions into it with `cvpilot eval`. Returns (run id, result)."""
    run_id = new_run(name)
    result = sh(
        f"cvpilot eval {run_id} --preds {preds} --gts {gts} --classes classes.txt --allow-closed {extra}",
        quiet=not show,
        keep=True,
        lines=lines,
        log_lines=2,
        **kw,
    )
    return run_id, result


def record(case, expected_code, expected_text, result, check=None, label=None):
    """Add a row: the exit code and a phrase we expected (from the docs), and what really happened.

    `check` is an optional test of the output text for expectations that are a property
    (for example "a number above 190") rather than a fixed phrase; `label` then names it."""
    everything = result.out + "\n" + result.err
    hit = next((ln for ln in everything.splitlines() if expected_text in ln), None)
    shown = log_summary(hit) if hit and hit.startswith("{") else (hit or "(phrase not found)")
    ok = result.code == expected_code and hit is not None and (check is None or check(everything))
    CASES.append(
        dict(
            case=case,
            expected=label or f"exit {expected_code}, mentions '{expected_text}'",
            actual=f"exit {result.code}, " + shown.strip()[:90],
            ok=ok,
        )
    )
    print(f"-> recorded: {'matches the expectation' if ok else 'DIFFERS from the expectation'}")


def py(source, name, env=None, **kw):
    """Write a small Python program to a file and run it in its own process."""
    Path(f"{name}.py").write_text(source.strip() + "\n")
    return sh(f"python {name}.py", keep=True, env=env, **kw)

detections at 640 px: 16017 (from the cache)
detections at 480 px: 14304 (from the cache)


**What this shows.** The real labels and real detections (640 px and 480 px) are ready. `evaluate()` creates a run and scores predictions into it with `cvpilot eval`; `record()` adds one row to the table at the end, built from the real exit code and output.

# Part A. Bad inputs to `cvpilot eval`

`cvpilot eval` reads three files: the ground truth, the predictions and the class list (see the first notebook for the format).
Bad numbers must stop here, loudly, because everything after trusts what `eval` stored.

### Case 1. What could go wrong: some prediction scores are `NaN`

`NaN` (not a number) compares false with everything, so a model that outputs it could slip through any check that uses `>` or `<`.

In [7]:
nan_preds = preds640.set_column(
    preds640.schema.get_field_index("score"),
    "score",
    pa.array(
        [float("nan") if i % 1000 == 0 else v for i, v in enumerate(preds640["score"].to_pylist())]
    ),
)
pq.write_table(nan_preds, "preds_nan.parquet")
print(f"{sum(v != v for v in nan_preds['score'].to_pylist())} of {len(nan_preds)} scores are NaN")
_, r = evaluate("nan-scores", "preds_nan.parquet")
record("Prediction scores contain NaN", 1, "score", r)

17 of 16017 scores are NaN


$ cvpilot eval 01M51WXDMR6846JMY9YKNGGNP0 --preds preds_nan.parquet --gts gt.parquet --classes classes.txt --allow-closed 
stderr: error: predictions have non-finite scores
log[ERROR] command_failed command=eval exception=Traceback (most recent call last):
  File "~
exit code: 1
-> recorded: matches the expectation


**What this shows.** `eval` refuses the whole file (exit 1) and names the column, instead of quietly ignoring or counting the bad rows. Nothing is stored for this run.

### Case 2. What could go wrong: a predicted class id is not in the class list

The list has 80 names (ids 0 to 79). A model trained with a different label map could output id 99.

In [8]:
cls = preds640["cls"].to_pylist()
cls[5] = 99
pq.write_table(
    preds640.set_column(preds640.schema.get_field_index("cls"), "cls", pa.array(cls, pa.int64())),
    "preds_badclass.parquet",
)
_, r = evaluate("bad-class-id", "preds_badclass.parquet")
record("Predicted class id outside the class list", 1, "outside [0, 80)", r)

$ cvpilot eval 01M51WXE94FCFN4FNAKTSVTHP4 --preds preds_badclass.parquet --gts gt.parquet --classes classes.txt --allow-closed 
stderr: error: predictions has class id 99 outside [0, 80)
log[ERROR] command_failed command=eval exception=Traceback (most recent call last):
  File "~
exit code: 1
-> recorded: matches the expectation


**What this shows.** Exit 1 with the offending id and the valid range. Silently mapping it to some class would have scored the model against the wrong labels.

### Case 3. What could go wrong: predictions for an image that is not in the ground truth

For example a file name mismatch between the two pipelines.

In [9]:
ghost = pa.table(
    {
        "image_id": ["not_in_ground_truth"],
        "cls": pa.array([0], pa.int64()),
        "score": [0.9],
        "x1": [10.0],
        "y1": [10.0],
        "x2": [50.0],
        "y2": [50.0],
    }
)
pq.write_table(pa.concat_tables([preds640, ghost.cast(preds640.schema)]), "preds_ghost.parquet")
_, r = evaluate("unknown-image", "preds_ghost.parquet")
record(
    "Predictions for an image that is not in the ground truth", 1, "not in the evaluation set", r
)

$ cvpilot eval 01M51WXF0ZTE1E4RHT1AP0TSW8 --preds preds_ghost.parquet --gts gt.parquet --classes classes.txt --allow-closed 
stderr: error: predictions refers to image id <19 characters, sha256:5fbfb353> that is not in the evaluation set (1 rows affected)
log[ERROR] command_failed command=eval exception=Traceback (most recent call last):
  File "~
exit code: 1
-> recorded: matches the expectation


**What this shows.** Exit 1. The message refers to the image id only by its length and a short fingerprint, because image ids are often file names or personal data and should not end up in logs.

### Case 4. What could go wrong: an image that has no objects

Two of the 128 photos have no labels at all. There are two ways to describe that, and only one is right.
In the ground truth file an image without objects is a row with an **empty** `cls` and empty box. If you simply leave the image out, the
evaluation set shrinks, and every detection the model makes on that image has nowhere to go.

In [10]:
empty_images = sorted(
    set(gt["image_id"].to_pylist())
    - {
        i
        for i, c in zip(gt["image_id"].to_pylist(), gt["cls"].to_pylist(), strict=True)
        if c is not None
    }
)
print("images without objects:", empty_images)
print(
    "detections the model made on them:",
    sum(i in empty_images for i in preds640["image_id"].to_pylist()),
)
_, r_ok = evaluate("empty-images-declared", "preds_640.parquet", lines=6)
record("Image without objects, declared as an empty row", 0, "n_images: 128", r_ok)
dropped = gt.filter(pc.invert(pc.is_in(gt["image_id"], pa.array(empty_images))))
pq.write_table(dropped, "gt_dropped.parquet")
_, r_bad = evaluate("empty-images-dropped", "preds_640.parquet", gts="gt_dropped.parquet")
record("Image without objects, left out of the ground truth", 1, "not in the evaluation set", r_bad)

images without objects: ['000000000250', '000000000508']
detections the model made on them: 27


$ cvpilot eval 01M51WXFP40HEP98N7HB4A6NTM --preds preds_640.parquet --gts gt.parquet --classes classes.txt --allow-closed 
run_id: 01M51WXFP40HEP98N7HB4A6NTM
eval_set_hash: b720c966844f83652fb422c7aac5587ae4a340258c5e4aca51ee9f60b5f68213
step: 0
n_images: 128
mAP50_95: 0.5033 [0.4692, 0.5657]
mAP50: 0.6711 [0.6286, 0.7319]
... (10 more lines)
log[INFO] eval_completed run_id=01M51WXFP40HEP98N7HB4A6NTM step=0 task=detection eval_set_hash=b720c966844f83652fb422c7aac5587ae4a340258c5e n_images=128 n_classes=80 n_predictions=16017 duration_ms=10
exit code: 0
-> recorded: matches the expectation


$ cvpilot eval 01M51WXHADNMNRQP93TM1Q0A69 --preds preds_640.parquet --gts gt_dropped.parquet --classes classes.txt --allow-closed 
stderr: error: predictions refers to image id <12 characters, sha256:765ec818> that is not in the evaluation set (27 rows affected)
log[ERROR] command_failed command=eval exception=Traceback (most recent call last):
  File "~
exit code: 1
-> recorded: matches the expectation


**What this shows.** Declared as an empty row the images stay in the evaluation set (128 images, exit 0): the detections on them count as false alarms, which is exactly what they are. Left out, the same predictions are refused (exit 1), so you cannot lose those false alarms by accident.

### Case 5. What could go wrong: no detections at all, or no file at all

A model that finds nothing is a *valid but terrible* result. A missing file is a *mistake*. They must not look alike.

In [11]:
pq.write_table(preds640.slice(0, 0), "preds_empty.parquet")
print("rows in the empty predictions file:", pq.read_table("preds_empty.parquet").num_rows)
_, r_empty = evaluate("zero-detections", "preds_empty.parquet", lines=4)
record("Zero detections (a valid, empty predictions file)", 0, "mAP50_95: 0.0000", r_empty)
_, r_missing = evaluate("missing-file", "no_such_file.parquet", lines=6)
record("Predictions file that does not exist", 2, "does not exist", r_missing)

rows in the empty predictions file: 0


$ cvpilot eval 01M51WXHVB25A5MFBWQDZB3T22 --preds preds_empty.parquet --gts gt.parquet --classes classes.txt --allow-closed 
run_id: 01M51WXHVB25A5MFBWQDZB3T22
eval_set_hash: b720c966844f83652fb422c7aac5587ae4a340258c5e4aca51ee9f60b5f68213
step: 0
n_images: 128
... (12 more lines)
log[INFO] eval_completed run_id=01M51WXHVB25A5MFBWQDZB3T22 step=0 task=detection eval_set_hash=b720c966844f83652fb422c7aac5587ae4a340258c5e n_images=128 n_classes=80 n_predictions=0 duration_ms=148.4
exit code: 0
-> recorded: matches the expectation


$ cvpilot eval 01M51WXJGGPG4N78D5EMVKX97K --preds no_such_file.parquet --gts gt.parquet --classes classes.txt --allow-closed 
stderr: Usage: cvpilot eval [OPTIONS] {run_id}
stderr: Try 'cvpilot eval --help' for help.
... (1 more log lines)
exit code: 2
-> recorded: matches the expectation


**What this shows.** An empty predictions file is accepted (exit 0) and scores zero: the model really did find nothing, and a later gate will fail it for being worse. A path that does not exist is a usage error (exit 2) and stores nothing.

# Part B. Comparing and gating

A shared setup: the 640 px detections are the baseline `prod`, the 480 px detections are the candidate (as in the first notebook).

In [12]:
PROD, r = evaluate("baseline-640", "preds_640.parquet", show=False)
CAND, r2 = evaluate("candidate-480", "preds_480.parquet", show=False)
if r.code or r2.code:
    raise RuntimeError("the shared baseline evaluations failed")
sh(f"cvpilot baseline set prod {PROD}", log_lines=0)

$ cvpilot baseline set prod 01M51WXJWG3Y9ZY9QXG14HTV9A
pinned prod: run 01M51WXJWG3Y9ZY9QXG14HTV9A eval_set b720c966844f step 0
... (1 more log lines)
exit code: 0


**What this shows.** Two real evaluations on the full 128-photo set, and the 640 px one is pinned as `prod`.

### Case 6. What could go wrong: the two models were evaluated on different photos

A candidate scored on 64 of the 128 photos is **not comparable** with a baseline scored on all of them: the numbers look alike but measure different things.

In [13]:
half = sorted(set(gt["image_id"].to_pylist()))[:64]
gt_half = gt.filter(pc.is_in(gt["image_id"], pa.array(half)))
preds_half = preds480.filter(pc.is_in(preds480["image_id"], pa.array(half)))
pq.write_table(gt_half, "gt_half.parquet")
pq.write_table(preds_half, "preds_half.parquet")
HALF, r = evaluate("candidate-half-set", "preds_half.parquet", gts="gt_half.parquet", show=False)
r = sh(f"cvpilot compare {HALF} --baseline tag:prod", keep=True, log_lines=2)
record("Compare two runs evaluated on different image sets", 1, "no snapshot for eval set", r)

$ cvpilot compare 01M51WXP6853WBNSN9V4NSZBN7 --baseline tag:prod
stderr: error: run 01M51WXP6853WBNSN9V4NSZBN7 has no snapshot for eval set b720c966844f
log[ERROR] compare_refused command=compare reason=eval_set_hash candidate_run_id=01M51WXP6853WBNSN9V4NSZBN7 baseline_run_id=01M51WXJWG3Y9ZY9QXG14HTV9A candidate_eval_set_hash=21fa19facd0499ed5a9957759
exit code: 1
-> recorded: matches the expectation


**What this shows.** `compare` refuses (exit 1) and says why: the candidate has no evaluation of the baseline's evaluation set. The evaluation set is identified by a fingerprint of the ground truth and class list, so two machines that evaluate the same photos agree on it.

### Case 7. What could go wrong: too few images to trust the result

On 20 photos one lucky detection can move a score by many points. By default the gate refuses to decide below 30 images. This fails closed instead of passing on too little data.

In [14]:
few = sorted(set(gt["image_id"].to_pylist()))[:20]
gt_few = gt.filter(pc.is_in(gt["image_id"], pa.array(few)))
pq.write_table(gt_few, "gt_few.parquet")
pq.write_table(
    preds640.filter(pc.is_in(preds640["image_id"], pa.array(few))), "preds_few_640.parquet"
)
pq.write_table(
    preds480.filter(pc.is_in(preds480["image_id"], pa.array(few))), "preds_few_480.parquet"
)
FEW0, _ = evaluate("few-baseline", "preds_few_640.parquet", gts="gt_few.parquet", show=False)
FEW1, _ = evaluate("few-candidate", "preds_few_480.parquet", gts="gt_few.parquet", show=False)
sh(f"cvpilot baseline set tiny {FEW0}", quiet=True)
r = sh(f"cvpilot gate {FEW1} --baseline tag:tiny --max-drop mAP50_95=50", keep=True, log_lines=2)
record("Gate on an evaluation set of 20 images (default minimum 30)", 1, "too_few_images", r)
r = sh(
    f"cvpilot gate {FEW1} --baseline tag:tiny --max-drop mAP50_95=50 --min-images 10",
    keep=True,
    log_lines=2,
)
record("Same 20 images with an explicit --min-images 10", 0, "gate: PASS", r)

$ cvpilot gate 01M51WXRZ0BX0S3Z79CXTK40YC --baseline tag:tiny --max-drop mAP50_95=50
gate: FAIL (1 violations, worst: [too_few_images] evaluation set) run 01M51WXRZ0BX0S3Z79CXTK40YC step 0 vs baseline tag:tiny
  - [too_few_images] evaluation set: the evaluation set has 20 images, fewer than --min-images 30
log[ERROR] gate_failed run_id=01M51WXRZ0BX0S3Z79CXTK40YC baseline_tag=tiny n_violations=1 worst_metric=evaluation set worst_kind=too_few_images worst_delta=None
exit code: 1
-> recorded: matches the expectation


$ cvpilot gate 01M51WXRZ0BX0S3Z79CXTK40YC --baseline tag:tiny --max-drop mAP50_95=50 --min-images 10
gate: PASS (1 checks, 0 slices skipped) run 01M51WXRZ0BX0S3Z79CXTK40YC step 0 vs baseline tag:tiny
log[INFO] gate_passed run_id=01M51WXRZ0BX0S3Z79CXTK40YC baseline_tag=tiny n_violations=0
exit code: 0
-> recorded: matches the expectation


**What this shows.** With the default minimum the gate exits 1 and says `too_few_images`, even though the drop is far inside the generous limit of 50 points. Lowering the minimum is possible, but it is an explicit decision you write into the command, not a silent default.

### Case 8. What could go wrong: a metric the gate was asked about does not exist

A typo in a metric name (`mAP50_96`), or a metric that an older snapshot never recorded, must not make the check disappear.

In [15]:
r = sh(f"cvpilot gate {CAND} --baseline tag:prod --max-drop mAP50_96=5", keep=True, log_lines=2)
record("Gate on a metric that is not in the snapshots", 1, "headline_missing", r)

$ cvpilot gate 01M51WXMDV0JD4F5FCGP75ESJ5 --baseline tag:prod --max-drop mAP50_96=5
gate: FAIL (1 violations, worst: [headline_missing] mAP50_96) run 01M51WXMDV0JD4F5FCGP75ESJ5 step 0 vs baseline tag:prod
  - [headline_missing] mAP50_96: metric mAP50_96 is absent from a snapshot
log[ERROR] gate_failed run_id=01M51WXMDV0JD4F5FCGP75ESJ5 baseline_tag=prod n_violations=1 worst_metric=mAP50_96 worst_kind=headline_missing worst_delta=None
exit code: 1
-> recorded: matches the expectation


**What this shows.** Exit 1 with `headline_missing`. If a limit that cannot be checked counted as a pass, a misspelled limit would turn the gate off without anyone noticing.

### Case 9. What could go wrong: the slice measurement exists on the baseline only

Slices (here: photos whose typical object is small or large) must have been measured the same way on both sides.

In [16]:
areas = {}
g = gt.to_pydict()
for i, c in enumerate(g["cls"]):
    if c is not None:
        share = (
            (g["x2"][i] - g["x1"][i]) * (g["y2"][i] - g["y1"][i]) / (g["width"][i] * g["height"][i])
        )
        areas.setdefault(g["image_id"][i], []).append(share)
typical = {
    k: float(np.median(v)) for k, v in areas.items()
}  # typical object size per image, as a share of the image
cut = float(np.median(list(typical.values())))
size_group = [
    "small_objects" if typical.get(i, 0.0) < cut else "large_objects" for i in g["image_id"]
]
pq.write_table(gt.append_column("object_size", pa.array(size_group)), "gt_sliced.parquet")
counts = {k: size_group.count(k) for k in sorted(set(size_group))}
print("photos are split by their typical object size (median share of the photo):", f"{cut:.3f}")
print("rows per group:", counts)
S0, _ = evaluate(
    "sliced-baseline-640",
    "preds_640.parquet",
    gts="gt_sliced.parquet",
    extra="--slices objects=object_size",
    show=False,
)
S1, _ = evaluate(
    "sliced-candidate-480",
    "preds_480.parquet",
    gts="gt_sliced.parquet",
    extra="--slices objects=object_size",
    show=False,
)
S2, _ = evaluate("unsliced-candidate-480", "preds_480.parquet", gts="gt_sliced.parquet", show=False)
sh(f"cvpilot baseline set sliced {S0}", quiet=True)
r = sh(
    f"cvpilot gate {S2} --baseline tag:sliced --slice-max-drop 'objects=*:3'",
    keep=True,
    log_lines=2,
)
record("Slice limit, slices measured on the baseline only", 1, "slice", r)

photos are split by their typical object size (median share of the photo): 0.050
rows per group: {'large_objects': 188, 'small_objects': 743}


$ cvpilot gate 01M51WXYHZ5NW0WBPYYTGSQBBS --baseline tag:sliced --slice-max-drop 'objects=*:3'
stderr: error: the images are sliced differently in the two snapshots (or only one records its slice membership); evaluate both with the same --slices
log[ERROR] compare_refused command=gate reason=slice_hash candidate_run_id=01M51WXYHZ5NW0WBPYYTGSQBBS baseline_run_id=01M51WXVH3X12T3C1G0JGKX4C4 candidate_eval_set_hash=b720c966844f83652fb422c7aac5587
exit code: 1
-> recorded: matches the expectation


**What this shows.** The candidate was scored on the same photos but without `--slices`, so there is nothing to compare for the slice limit and the gate exits 1 instead of skipping the check. Fix: evaluate both sides with the same `--slices` option.

### Case 10. What could go wrong: the average hides a per-class regression

This is the main reason CV-Pilot exists. The 480 px candidate loses only a few points on average, so a limit on the average is happy,
while some classes lose a lot (see the first notebook for the list).

In [17]:
r = sh(f"cvpilot gate {CAND} --baseline tag:prod --max-drop mAP50_95=5", keep=True, log_lines=1)
record("Average-only gate on a candidate with per-class losses", 0, "gate: PASS", r)
r = sh(
    f"cvpilot gate {CAND} --baseline tag:prod --max-drop mAP50_95=5 --per-class-max-drop AP50_95=20",
    keep=True,
    log_lines=1,
    lines=8,
)
record("Same candidate with a per-class limit of 20 points", 1, "[class_drop]", r)

$ cvpilot gate 01M51WXMDV0JD4F5FCGP75ESJ5 --baseline tag:prod --max-drop mAP50_95=5
gate: PASS (1 checks, 0 slices skipped) run 01M51WXMDV0JD4F5FCGP75ESJ5 step 0 vs baseline tag:prod
log[INFO] gate_passed run_id=01M51WXMDV0JD4F5FCGP75ESJ5 baseline_tag=prod n_violations=0
exit code: 0
-> recorded: matches the expectation


$ cvpilot gate 01M51WXMDV0JD4F5FCGP75ESJ5 --baseline tag:prod --max-drop mAP50_95=5 --per-class-max-drop AP50_95=20
gate: FAIL (6 violations, worst: [class_drop] sandwich) run 01M51WXMDV0JD4F5FCGP75ESJ5 step 0 vs baseline tag:prod
  - [class_drop] sandwich: AP50_95 of class dropped 41.42 points (limit 20)
  - [class_drop] microwave: AP50_95 of class dropped 37.82 points (limit 20)
  - [class_drop] horse: AP50_95 of class dropped 26.67 points (limit 20)
  - [class_drop] skis: AP50_95 of class dropped 25.00 points (limit 20)
  - [class_drop] refrigerator: AP50_95 of class dropped 22.99 points (limit 20)
  - [class_drop] laptop: AP50_95 of class dropped 22.77 points (limit 20)
log[ERROR] gate_failed run_id=01M51WXMDV0JD4F5FCGP75ESJ5 baseline_tag=prod n_violations=6 worst_metric=sandwich worst_kind=class_drop worst_delta=-41.419141914
exit code: 1
-> recorded: matches the expectation


**What this shows.** The average-only gate passes (exit 0); adding the per-class limit makes the same candidate fail (exit 1) and lists the classes. Both numbers come from the same stored evaluation.

### Case 11. What could go wrong: a slice regression (small objects vs large objects)

Same idea for groups of photos instead of classes. Here the slices are the two kinds of photo defined above. The candidate loses
about 4 points on photos with small objects and about 1 point on photos with large objects.

In [18]:
r = sh(f"cvpilot gate {S1} --baseline tag:sliced --max-drop mAP50_95=5", keep=True, log_lines=1)
record("Average-only gate, slices measured on both sides", 0, "gate: PASS", r)
r = sh(
    f"cvpilot gate {S1} --baseline tag:sliced --max-drop mAP50_95=5 --slice-max-drop 'objects=*:3'",
    keep=True,
    log_lines=1,
)
record("Slice limit of 3 points for every slice", 1, "[slice_drop] objects=small_objects", r)

$ cvpilot gate 01M51WXX305XXMK9JYXM0805M4 --baseline tag:sliced --max-drop mAP50_95=5
gate: PASS (1 checks, 0 slices skipped) run 01M51WXX305XXMK9JYXM0805M4 step 0 vs baseline tag:sliced
log[INFO] gate_passed run_id=01M51WXX305XXMK9JYXM0805M4 baseline_tag=sliced n_violations=0
exit code: 0
-> recorded: matches the expectation


$ cvpilot gate 01M51WXX305XXMK9JYXM0805M4 --baseline tag:sliced --max-drop mAP50_95=5 --slice-max-drop 'objects=*:3'
gate: FAIL (1 violations, worst: [slice_drop] objects=small_objects) run 01M51WXX305XXMK9JYXM0805M4 step 0 vs baseline tag:sliced
  - [slice_drop] objects=small_objects: mAP50_95 of slice dropped 3.75 points (limit 3)
log[ERROR] gate_failed run_id=01M51WXX305XXMK9JYXM0805M4 baseline_tag=sliced n_violations=1 worst_metric=objects=small_objects worst_kind=slice_drop worst_delta=-3.749672475
exit code: 1
-> recorded: matches the expectation


**What this shows.** With `--slice-max-drop 'objects=*:3'` (any slice may drop at most 3 points) the gate fails on the small-object photos only: the group the average hid. The large-object photos stayed inside the limit.

### Case 12. What could go wrong: a baseline that is a run that is still running

A run that has not finished can still change (or die). We start a run in another process and keep it open, evaluate into it, and then try to use it as the baseline.

In [19]:
HOLD = r"""
import sys
import cvpilot
run = cvpilot.init("coco128-yolo", task="detection", name="still-running")
print(run.run_id, flush=True)
sys.stdin.read()  # stay alive (and 'running') until the notebook closes our input
"""
Path("hold_run.py").write_text(HOLD)
holder = subprocess.Popen(
    [sys.executable, "hold_run.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True,
    cwd=WORK,
    env={**os.environ},
)  # noqa: S603
LIVE = holder.stdout.readline().strip()
sh(
    f"cvpilot eval {LIVE} --preds preds_640.parquet --gts gt.parquet --classes classes.txt",
    quiet=True,
    expect=0,
)
sh(f"cvpilot runs show {LIVE}", lines=4, log_lines=0)
r = sh(f"cvpilot gate {CAND} --baseline {LIVE} --max-drop mAP50_95=5", keep=True, log_lines=1)
record("Gate against a run that is still running, given as a run id", 1, "must be finished", r)

$ cvpilot runs show 01M51WY2F87NXWQDJ4PHPX0JQP
run_id: 01M51WY2F87NXWQDJ4PHPX0JQP
project: coco128-yolo
name: still-running
status: running
... (23 more lines)
exit code: 0


$ cvpilot gate 01M51WXMDV0JD4F5FCGP75ESJ5 --baseline 01M51WY2F87NXWQDJ4PHPX0JQP --max-drop mAP50_95=5
stderr: error: baseline run 01M51WY2F87NXWQDJ4PHPX0JQP is running; a baseline run must be finished
... (1 more log lines)
exit code: 1
-> recorded: matches the expectation


**What this shows.** The run is `running`, yet it already has an evaluation. Using it as a baseline by run id is refused (exit 1): a baseline must be a finished run.

In [20]:
r = sh(f"cvpilot baseline set live-pin {LIVE}", keep=True, log_lines=1)
record("Pinning a run that is still running with baseline set", 1, "must be finished", r)
sh("cvpilot baseline list", log_lines=0)
holder.stdin.close()
holder.wait(timeout=60)

$ cvpilot baseline set live-pin 01M51WY2F87NXWQDJ4PHPX0JQP
stderr: error: run 01M51WY2F87NXWQDJ4PHPX0JQP is running; a baseline run must be finished
... (1 more log lines)
exit code: 1
-> recorded: matches the expectation


$ cvpilot baseline list
TAG                               RUN_ID                           STEP  EVAL_SET      PINNED_AT
prod                              01M51WXJWG3Y9ZY9QXG14HTV9A          0  b720c966844f  2026-10-16T08:23:11.785285+00:00
sliced                            01M51WXVH3X12T3C1G0JGKX4C4          0  b720c966844f  2026-10-16T08:23:22.134980+00:00
tiny                              01M51WXQSYF5HRMM2SKMJ7MPSN          0  c5693cf0fb59  2026-10-16T08:23:16.057811+00:00
exit code: 0


0

**What this shows.** Pinning the same unfinished run as a named baseline is refused the same way (exit 1) and nothing is pinned (`baseline list` has no `live-pin`). There is no override: a run whose process died is first marked failed with `recover_run`, and you pin a run that finished. (An earlier version of CV-Pilot accepted this pin; that inconsistency was found while writing this notebook and fixed.)

### Case 13. What could go wrong: a run is its own baseline

Comparing a model with itself always looks perfect.

In [21]:
r = sh(f"cvpilot gate {CAND} --baseline {CAND} --max-drop mAP50_95=5", keep=True, log_lines=1)
record("A run used as its own baseline", 1, "its own baseline", r)

$ cvpilot gate 01M51WXMDV0JD4F5FCGP75ESJ5 --baseline 01M51WXMDV0JD4F5FCGP75ESJ5 --max-drop mAP50_95=5
stderr: error: a run cannot be its own baseline
... (1 more log lines)
exit code: 1
-> recorded: matches the expectation


**What this shows.** Refused (exit 1) before any number is looked at.

### Case 14. What could go wrong: an evaluation result edited by hand after the fact

We take the 160 px model (a clearly worse one), let the gate fail it honestly, then edit its stored result so that it claims the baseline's score.
Without a check, the edit would turn a red build green.

In [22]:
pq.write_table(detect(160, use_cache=True)[0], "preds_160.parquet")
BAD, _ = evaluate("candidate-160", "preds_160.parquet", show=False)
sh(f"cvpilot gate {BAD} --baseline tag:prod --max-drop mAP50_95=5", log_lines=1, lines=3, expect=1)
file = next((Path(os.environ["CVPILOT_STORE_DIR"]) / "coco128-yolo" / "evals" / BAD).glob("*.json"))
data = json.loads(file.read_text())
print(
    f"stored mAP50_95 {data['metrics']['mAP50_95']:.4f}  ->  edited to 0.5033 (the baseline's value)"
)
data["metrics"]["mAP50_95"] = 0.5033
file.chmod(0o644)
file.write_text(json.dumps(data))
r = sh(f"cvpilot gate {BAD} --baseline tag:prod --max-drop mAP50_95=5", keep=True, log_lines=1)
record("Evaluation file edited by hand to look better", 1, "digest mismatch", r)

$ cvpilot gate 01M51WY5EB6DC0VZD8EWEKCFN6 --baseline tag:prod --max-drop mAP50_95=5
gate: FAIL (1 violations, worst: [headline_drop] mAP50_95) run 01M51WY5EB6DC0VZD8EWEKCFN6 step 0 vs baseline tag:prod
  - [headline_drop] mAP50_95: mAP50_95 dropped 32.15 points (limit 5)
log[ERROR] gate_failed run_id=01M51WY5EB6DC0VZD8EWEKCFN6 baseline_tag=prod n_violations=1 worst_metric=mAP50_95 worst_kind=headline_drop worst_delta=-32.147019628
exit code: 1
stored mAP50_95 0.1819  ->  edited to 0.5033 (the baseline's value)


$ cvpilot gate 01M51WY5EB6DC0VZD8EWEKCFN6 --baseline tag:prod --max-drop mAP50_95=5
stderr: error: snapshot file b720c966844f_000000000.json of run 01M51WY5EB6DC0VZD8EWEKCFN6: digest mismatch
... (1 more log lines)
exit code: 1
-> recorded: matches the expectation


**What this shows.** Before the edit the gate fails the model for the right reason (a 32-point drop). After the edit it still fails, but now because the stored file no longer matches its fingerprint (`digest mismatch`). The tampered number was never used. (This protects against accidents and casual edits; someone who can rewrite both the file and the database can recompute the fingerprints. CI should recompute from trusted ground truth: see `docs/guide/trust.md`.)

# Part C. Checkpoints that are damaged, missing or tampered with

For these cases the job is the real classifier from the first notebook. First we need crops, the training script and a job that was killed so that it
left checkpoints behind.

In [23]:
import zlib


def build_crops(gt_table, min_side=12, min_per_class=20, cap=60):
    cols = gt_table.to_pydict()
    rows = [
        i
        for i, c in enumerate(cols["cls"])
        if c is not None
        and cols["x2"][i] - cols["x1"][i] >= min_side
        and cols["y2"][i] - cols["y1"][i] >= min_side
    ]
    counts = {}
    for i in rows:
        counts[cols["cls"][i]] = counts.get(cols["cls"][i], 0) + 1
    keep = sorted((c for c, n in counts.items() if n >= min_per_class), key=lambda c: -counts[c])
    opened, taken, xs, ys, ids = {}, {}, [], [], []
    for i in rows:
        c = cols["cls"][i]
        if c not in keep or taken.get(c, 0) >= cap:
            continue
        image_id = cols["image_id"][i]
        if image_id not in opened:
            opened[image_id] = Image.open(
                COCO_ROOT / "images" / "train2017" / f"{image_id}.jpg"
            ).convert("RGB")
        box = (cols["x1"][i], cols["y1"][i], cols["x2"][i], cols["y2"][i])
        xs.append(np.asarray(opened[image_id].crop(box).resize((32, 32))))
        ys.append(keep.index(c))
        ids.append(image_id)
        taken[c] = taken.get(c, 0) + 1
    val = np.array([zlib.crc32(s.encode()) % 5 == 0 for s in ids])
    return np.stack(xs).transpose(0, 3, 1, 2).astype(np.uint8), np.array(ys), val


x, y, is_val = build_crops(gt)
np.savez("crops.npz", x=x, y=y, val=is_val)
print(f"{len(y)} real object crops, {int(is_val.sum())} held out for validation")

215 real object crops, 26 held out for validation


In [24]:
TRAIN_SCRIPT = r"""
# train_clf.py: a tiny image classifier trained on real COCO128 object crops.
# With --checkpoints it saves a crash-proof checkpoint every epoch and resumes by itself.
import argparse
import sys
import time

import numpy as np
import torch
from torch import nn

import cvpilot

parser = argparse.ArgumentParser()
parser.add_argument("--project", required=True)
parser.add_argument("--epochs", type=int, default=5)
parser.add_argument("--checkpoints", action="store_true", help="save and resume checkpoints")
parser.add_argument("--progress", action="store_true", help="print a line every 5 batches")
parser.add_argument("--pause-ms", type=int, default=0, help="wait this long after every batch")
args = parser.parse_args()

BATCH, PASSES = 32, 16  # one epoch = 16 shuffled passes over the (few) training crops
data = np.load("crops.npz")
images = torch.tensor(data["x"]).float() / 255
labels = torch.tensor(data["y"])
train_idx, val_idx = np.where(~data["val"])[0], np.where(data["val"])[0]
n_batches = -(-len(train_idx) * PASSES // BATCH)  # batches per epoch (rounded up)


def block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out),
                         nn.ReLU(), nn.MaxPool2d(2))


torch.manual_seed(0)
model = nn.Sequential(block(3, 16), block(16, 32), block(32, 64), nn.Flatten(),
                      nn.Dropout(0.3), nn.Linear(64 * 4 * 4, int(labels.max()) + 1))
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)


def epoch_order(epoch):  # the same shuffle for the same epoch, so a resumed run sees the same data
    rng = np.random.default_rng(1000 + epoch)
    return np.concatenate([rng.permutation(train_idx) for _ in range(PASSES)])


def val_accuracy():
    model.eval()
    with torch.no_grad():
        return (model(images[val_idx]).argmax(1) == labels[val_idx]).float().mean().item()


point = None
if args.checkpoints:
    from cvpilot.ckpt import CheckpointManager, apply_rng, find_resume_point
    from cvpilot.runs import recover_run

    point = find_resume_point(args.project)  # newest run of this project that can be resumed
    if point is not None and point.needs_recover:  # its process died: mark it failed first
        recover_run(args.project, point.run_id)
run = cvpilot.init(
    args.project, task="classification", seed=0,
    config={"epochs": args.epochs, "batch": BATCH, "lr": 2e-3, "model": "3-block CNN"},
    resume_from=None if point is None else point.run_id,
)
ckpt = None
epoch0, batch0 = 0, 0  # where to start
if args.checkpoints:
    ckpt = CheckpointManager(run, every_n_steps=n_batches, keep_last=3)  # one save per epoch
    restored = ckpt.restore()  # None on a fresh start
    if restored is None:
        print("starting fresh: no checkpoint to resume from", flush=True)
    else:
        state = restored.state
        model.load_state_dict({k: torch.from_numpy(v) for k, v in state["model"].items()})
        apply_rng(state["rng"])
        epoch0, batch0 = state["dataloader_state"]["epoch"], state["dataloader_state"]["batch"] + 1
        if batch0 >= n_batches:  # the checkpoint was the last batch of an epoch: start the next one
            epoch0, batch0 = epoch0 + 1, 0
        print(f"resumed from step {restored.manifest.step} (epoch {epoch0 + 1}, batch {batch0})",
              flush=True)

for epoch in range(epoch0, args.epochs):
    order = epoch_order(epoch)
    model.train()
    for b in range(batch0 if epoch == epoch0 else 0, n_batches):
        idx = order[b * BATCH:(b + 1) * BATCH]
        xb = images[idx]
        flip = torch.rand(len(idx), generator=torch.Generator().manual_seed(epoch * 10007 + b)) < 0.5
        xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
        loss = nn.functional.cross_entropy(model(xb), labels[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        time.sleep(args.pause_ms / 1000)  # a stand-in for a bigger model that takes longer per batch
        step = epoch * n_batches + b + 1  # global batch counter, 1-based
        cvpilot.log({"train/loss": loss.item()}, step=step)
        if args.progress and b % 5 == 0:
            print(f"epoch {epoch + 1}/{args.epochs} batch {b}/{n_batches} loss {loss.item():.3f}",
                  flush=True)
        if ckpt is not None:
            ckpt.maybe_save(step, model=model, rng=True,
                            dataloader_state={"epoch": epoch, "batch": b})
            if ckpt.preempted():  # SIGTERM arrived: the checkpoint was just written
                print(f"preempted at epoch {epoch + 1}, batch {b}: checkpoint saved", flush=True)
                ckpt.close()
                sys.exit(0)
    acc = val_accuracy()
    model.train()
    cvpilot.log({"val/accuracy": acc}, step=epoch * n_batches + n_batches)
    print(f"epoch {epoch + 1}/{args.epochs} done: loss {loss.item():.3f} val_accuracy {acc:.2f}",
          flush=True)
run.finish()
if ckpt is not None:
    ckpt.close()
print("run", run.run_id, "finished")
"""
Path("train_clf.py").write_text(TRAIN_SCRIPT)
print(
    len(TRAIN_SCRIPT.splitlines()),
    "lines written to train_clf.py (the same script as in the first notebook)",
)

115 lines written to train_clf.py (the same script as in the first notebook)


**What this shows.** Real crops of 7 kinds of objects and the same training script as in the first notebook: a PyTorch loop that logs with `cvpilot.log`, saves a checkpoint at the end of every epoch with `CheckpointManager` and resumes by itself.

In [25]:
def start_training(project, epochs=6, pause_ms=0, env=None):
    return subprocess.Popen(  # noqa: S603
        [
            sys.executable,
            "train_clf.py",
            "--project",
            project,
            "--epochs",
            str(epochs),
            "--checkpoints",
            "--progress",
            "--pause-ms",
            str(pause_ms),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        cwd=WORK,
        env={**os.environ, "CVPILOT_LOG_LEVEL": "INFO", **(env or {})},
    )


def watch(proc, until, sig):
    """Echo the job's progress lines; when `until(line)` is true send it the signal `sig`."""
    seen = []
    for line in proc.stdout:
        seen.append(line.rstrip())
        if until(line):
            proc.send_signal(sig)
            break
    proc.wait()
    return seen, proc.stderr.read()


def at(epoch, batch):
    key = f"epoch {epoch}/"
    return lambda line: (
        line.startswith(key)
        and " batch " in line
        and int(line.split("batch ")[1].split("/")[0]) >= batch
    )


def fork(tag):
    """A private copy of the store and of the checkpoints, so one case cannot disturb another."""
    shutil.copytree("store", f"{tag}/store")
    shutil.copytree("ckpt", f"{tag}/ckpt")
    return {"CVPILOT_STORE_DIR": f"{tag}/store", "CVPILOT_CHECKPOINT_DIR": f"{tag}/ckpt"}


with timed("training job killed in epoch 4"):
    seen, _ = watch(start_training("ck-base"), at(4, 5), signal.SIGKILL)
print("\n".join(seen[-2:]))
sh("cvpilot ckpt list ckpt --project ck-base", log_lines=0)

[training job killed in epoch 4: 4.8 s]
epoch 4/6 batch 0/95 loss 0.006
epoch 4/6 batch 5/95 loss 0.015


$ cvpilot ckpt list ckpt --project ck-base
01M51WY8KWJCT6KNSMG3G767TW  last  000000285  valid    anchored
01M51WY8KWJCT6KNSMG3G767TW  last  000000190  valid    anchored
01M51WY8KWJCT6KNSMG3G767TW  last  000000095  valid    anchored
exit code: 0


**What this shows.** A real job was killed with `SIGKILL` early in epoch 4. It left three checkpoints (ends of epochs 1, 2 and 3) behind. Each case below works on its own private copy of this state.

In [26]:
base_steps = sorted(
    int(p.stem.split("_")[1]) for p in Path("ckpt").glob("*/last/step_*.safetensors")
)
newest, previous, oldest = base_steps[-1], base_steps[-2], base_steps[0]
print("checkpoint steps:", base_steps)

checkpoint steps: [95, 190, 285]


### Case 15. What could go wrong: a checkpoint file is cut short

A copy that was interrupted, a disk error, or someone with a text editor. We truncate the newest file to 100 bytes.

In [27]:
E = fork("c15")
newest_file = next(Path(E["CVPILOT_CHECKPOINT_DIR"]).glob(f"*/last/step_{newest:09d}.safetensors"))
newest_file.chmod(0o644)
newest_file.write_bytes(newest_file.read_bytes()[:100])
print(f"truncated the checkpoint of step {newest} to 100 bytes")
r = sh("cvpilot ckpt verify c15/ckpt --project ck-base", keep=True, env=E, log_lines=0)
record("Newest checkpoint truncated: verify", 1, "size_mismatch", r)
r = py(
    """
from cvpilot.ckpt import find_resume_point

point = find_resume_point("ck-base")
print("resume point: step", point.manifest.step, "| the run was killed, so it needs recovery:", point.needs_recover)
""",
    "resume_after_truncation",
    env=E,
    log_lines=3,
)
record("Newest checkpoint truncated: resume point", 0, f"resume point: step {previous}", r)

truncated the checkpoint of step 285 to 100 bytes


$ cvpilot ckpt verify c15/ckpt --project ck-base
01M51WY8KWJCT6KNSMG3G767TW  last  000000285  corrupt  size_mismatch
01M51WY8KWJCT6KNSMG3G767TW  last  000000190  valid    anchored
01M51WY8KWJCT6KNSMG3G767TW  last  000000095  valid    anchored
... (4 more log lines)
exit code: 1
-> recorded: matches the expectation


$ python resume_after_truncation.py
resume point: step 190 | the run was killed, so it needs recovery: True
log[WARNING] checkpoint_skipped_corrupt run_id=01M51WY8KWJCT6KNSMG3G767TW step=285 kind=last reason=size_mismatch
log[INFO] resume_point_selected run_id=01M51WY8KWJCT6KNSMG3G767TW checkpoint_run_id=01M51WY8KWJCT6KNSMG3G767TW step=190 status=running trust_level=anchored reason=newest resumable run
exit code: 0
-> recorded: matches the expectation


**What this shows.** `ckpt verify` re-hashes every file and exits 1 for the truncated one (`size_mismatch`) while the others stay `valid`. A restarted job does not crash and does not load the broken file: `find_resume_point` skips it (the `checkpoint_skipped_corrupt` log line) and hands back the previous checkpoint, so you lose one epoch instead of the whole run.

### Case 16. What could go wrong: the newest checkpoints were deleted

A cleanup script, a full disk that was cleaned the wrong way, or someone rolling back to an older state. The remaining files are all genuine, but older than the progress that was recorded in the run store.

In [28]:
E = fork("c16")
for step in (newest, previous):
    for suffix in (".safetensors", ".manifest.json"):
        for f in Path(E["CVPILOT_CHECKPOINT_DIR"]).glob(f"*/last/step_{step:09d}{suffix}"):
            f.unlink()
print(f"deleted the checkpoints of steps {previous} and {newest}; only step {oldest} is left")
r = sh("cvpilot ckpt verify c16/ckpt --project ck-base", keep=True, env=E, log_lines=0)
record("Newest checkpoints deleted: verify with the default trust policy (any)", 0, "valid", r)
record("Default policy: a note says a newer step was recorded", 0, "was recorded", r)
r = sh(
    "cvpilot ckpt verify c16/ckpt --project ck-base --require-trust anchored",
    keep=True,
    env=E,
    log_lines=0,
)
record("Newest checkpoints deleted: verify with --require-trust anchored", 1, "rolled_back", r)
r = sh(
    f"cvpilot ckpt verify c16/ckpt --project ck-base --require-trust anchored --accept-rollback-to {oldest}",
    keep=True,
    env=E,
    log_lines=1,
)
record(f"Rollback accepted on purpose (--accept-rollback-to {oldest})", 0, "valid", r)

deleted the checkpoints of steps 190 and 285; only step 95 is left


$ cvpilot ckpt verify c16/ckpt --project ck-base
01M51WY8KWJCT6KNSMG3G767TW  last  000000095  valid    anchored
... (2 more log lines)
exit code: 0
-> recorded: matches the expectation
-> recorded: matches the expectation


$ cvpilot ckpt verify c16/ckpt --project ck-base --require-trust anchored
01M51WY8KWJCT6KNSMG3G767TW  last  000000095  corrupt  rolled_back
... (1 more log lines)
exit code: 1
-> recorded: matches the expectation


$ cvpilot ckpt verify c16/ckpt --project ck-base --require-trust anchored --accept-rollback-to 95
01M51WY8KWJCT6KNSMG3G767TW  last  000000095  valid    anchored
log[WARNING] checkpoint_rollback_accepted run_id=01M51WY8KWJCT6KNSMG3G767TW step=95 kind=last floor=285 accepted_to=95
... (1 more log lines)
exit code: 0
-> recorded: matches the expectation


**What this shows.** With the default policy (`any`) `verify` only judges the files, and the surviving old checkpoint is simply `valid` (exit 0): `docs/guide/trust.md` says that under `any` rollback is not enforced. With `--require-trust anchored` the run store's memory that a later step existed is enforced: the old checkpoint is reported `rolled_back` and `verify` exits 1, because the files are intact but progress is missing and silently restarting from step 95 would hide that. Going back on purpose is explicit (`--accept-rollback-to`, logged). To make that visible without changing the exit code, plain `verify` prints one `note:` line on stderr saying that the run store recorded a newer step than the newest valid checkpoint. (My first expectation was that plain `verify` would already report `rolled_back`, as the old help text suggested; the help text was misleading and has been rewritten.)

In [29]:
r = py(
    """
from cvpilot.ckpt import find_resume_point
from cvpilot.core.errors import CheckpointUntrustedError

try:
    find_resume_point("ck-base", require_trust="anchored")
except CheckpointUntrustedError as exc:
    print("refused:", exc.reason, "-", str(exc)[:150])
point = find_resume_point("ck-base", require_trust="anchored", accept_rollback_to=ACCEPT)
print("after accepting the rollback: resume from step", point.manifest.step)
""".replace("ACCEPT", str(oldest)),
    "rollback_api",
    env=E,
    log_lines=0,
)
record("Rollback through the Python API under a strict policy", 0, "refused: rolled_back", r)

$ python rollback_api.py
refused: rolled_back - checkpoints were rolled back: step 285 is known to have existed and nothing at or above it verifies (restore it, or accept falling back on purpose wit
after accepting the rollback: resume from step 95
... (4 more log lines)
exit code: 0
-> recorded: matches the expectation


**What this shows.** With `require_trust='anchored'` (every checkpoint must also be recorded in the run store) `find_resume_point` raises instead of returning the old file, and `accept_rollback_to=` is the explicit way through. With the default policy `any` no error is raised for a missing newest checkpoint; use the strict policy for shared or bucket-backed directories.

### Case 17. What could go wrong: someone plants a checkpoint with a huge step number

If a save only had to beat "the highest step in the folder", a planted `step_999999` would block every future save of this job. Here the job is running in this
process, and a well-formed checkpoint at step 999999 appears in its folder (made with internal helpers that stand in for an attacker who can write the folder).

In [30]:
E = {"CVPILOT_STORE_DIR": "c17/store", "CVPILOT_CHECKPOINT_DIR": "c17/ckpt"}
r = py(
    """
import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np

import cvpilot
from cvpilot.ckpt import CheckpointManager
from cvpilot.ckpt.directory import RunCheckpoints  # internal: only used to play the attacker
from cvpilot.ckpt.serializers import get_serializer
from cvpilot.core.errors import CheckpointError

root = Path(os.environ["CVPILOT_CHECKPOINT_DIR"])
run = cvpilot.init("ck-plant", task="classification")
ckpt = CheckpointManager(run, every_n_steps=1, keep_last=10, install_signal_handlers=False)
for step in (1, 2, 3):
    ckpt.save_now(step, model={"w": np.full(4, float(step), dtype=np.float32)})
print("saved steps 1, 2, 3")

forged = get_serializer("safetensors").dumps({"model": {"w": np.zeros(4, dtype=np.float32)}, "step": 999999})
RunCheckpoints(Path("forger"), run.run_id, create=True).save(
    kind="last", step=999999, data=forged, format="safetensors", versions={}, dataset_hash="merkle:" + "c" * 64
)
for name in ("step_000999999.safetensors", "step_000999999.manifest.json"):
    shutil.copy(Path("forger") / run.run_id / "last" / name, root / run.run_id / "last" / name)
print("planted a checkpoint at step 999999")

def cli(*args):
    code = "import sys; sys.argv[0] = 'cvpilot'; from cvpilot.cli.app import main; main()"
    done = subprocess.run([sys.executable, "-c", code, *args], capture_output=True, text=True)
    return done.stdout.strip().replace(str(root), "ckpt")

try:
    ckpt.save_now(4, model={"w": np.full(4, 4.0, dtype=np.float32)})
    print("step 4 was saved (unexpected)")
except CheckpointError as exc:
    print("save refused:", type(exc).__name__, "-", str(exc)[:120])
print("planted files still in place:", (root / run.run_id / "last" / "step_000999999.manifest.json").exists())
print("quarantine list:", cli("ckpt", "quarantine", str(root), "--run", run.run_id))
print("release:", cli("ckpt", "quarantine", str(root), "--run", run.run_id, "--clear"))
ckpt.save_now(4, model={"w": np.full(4, 4.0, dtype=np.float32)})
print("after the operator released the entry, step 4 saved")
ckpt.close()
run.finish()
""",
    "plant",
    env=E,
    log_lines=0,
)
record(
    "Planted checkpoint with step 999999: the save is refused",
    0,
    "save refused: CheckpointStepError",
    r,
)
record("Planted checkpoint: saving works after the operator clears the list", 0, "step 4 saved", r)

$ python plant.py
saved steps 1, 2, 3
planted a checkpoint at step 999999
save refused: CheckpointStepError - step 4 does not exceed 999999: an untrusted manifest at step 999999 is quarantined (listed, not moved) and still counts 
planted files still in place: True
quarantine list: 01M51WYEFSS8FFWG3Y4D5R9JHC  last  000999999  quarantined
release: 01M51WYEFSS8FFWG3Y4D5R9JHC  last  000999999  quarantined
released 1 quarantined manifest(s); no file was touched
after the operator released the entry, step 4 saved
... (12 more log lines)
exit code: 0
-> recorded: matches the expectation
-> recorded: matches the expectation


**What this shows.** The planted file is **reported** (it appears in the quarantine list), nothing is deleted or moved, and the next save is refused with a clear error (`CheckpointStepError`) instead of writing something inconsistent. The entry keeps blocking until a person looks at it and releases it with `cvpilot ckpt quarantine --clear`; after that, saving continues. Removing the planted files is a separate, confirmed command (`ckpt trash --yes`).

### Case 18. What could go wrong: a checkpoint is saved or read with the wrong key, or no key

With `CVPILOT_CKPT_KEY` set, every checkpoint carries a signature that only the holder of the key can make. We write checkpoints with a random key (generated here, kept in an environment variable of the child process and never printed).

In [31]:
import secrets

KEY = secrets.token_hex(32)
OTHER_KEY = secrets.token_hex(32)
with timed("authenticated training (3 epochs)"):
    sh(
        "python train_clf.py --project ck-auth --epochs 3 --checkpoints",
        env={"CVPILOT_CKPT_KEY": KEY},
        lines=2,
        log_lines=0,
        expect=0,
    )
AUTH_RUN = sh("cvpilot runs list --project ck-auth --json", quiet=True).json()[0]["run_id"]
verify = (
    f"cvpilot ckpt verify ckpt --run {AUTH_RUN} --project ck-auth --require-trust authenticated"
)
r = sh(verify, keep=True, env={"CVPILOT_CKPT_KEY": KEY}, log_lines=0, lines=3)
record("Authenticated checkpoints verified with the right key", 0, "authenticated", r)
r = sh(verify, keep=True, log_lines=1, lines=3)
record("Authenticated checkpoints, no key at all (a configuration error)", 2, "needs the key", r)
r = sh(verify, keep=True, env={"CVPILOT_CKPT_KEY": OTHER_KEY}, log_lines=1, lines=3)
record("Authenticated checkpoints, a different key", 1, "key_changed", r)

$ python train_clf.py --project ck-auth --epochs 3 --checkpoints
starting fresh: no checkpoint to resume from
epoch 1/3 done: loss 0.128 val_accuracy 0.35
... (3 more lines)
... (9 more log lines)
exit code: 0
[authenticated training (3 epochs): 5.1 s]


$ cvpilot ckpt verify ckpt --run 01M51WYGHQ6903KHS5WPAS25AC --project ck-auth --require-trust authenticated
01M51WYGHQ6903KHS5WPAS25AC  last  000000285  valid    authenticated
01M51WYGHQ6903KHS5WPAS25AC  last  000000190  valid    authenticated
01M51WYGHQ6903KHS5WPAS25AC  last  000000095  valid    authenticated
... (3 more log lines)
exit code: 0
-> recorded: matches the expectation


$ cvpilot ckpt verify ckpt --run 01M51WYGHQ6903KHS5WPAS25AC --project ck-auth --require-trust authenticated
stderr: error: require_trust=authenticated needs the key in CVPILOT_CKPT_KEY (never stored)
exit code: 2
-> recorded: matches the expectation


$ cvpilot ckpt verify ckpt --run 01M51WYGHQ6903KHS5WPAS25AC --project ck-auth --require-trust authenticated
01M51WYGHQ6903KHS5WPAS25AC  last  000000285  corrupt  key_changed
01M51WYGHQ6903KHS5WPAS25AC  last  000000190  corrupt  key_changed
01M51WYGHQ6903KHS5WPAS25AC  last  000000095  corrupt  key_changed
log[INFO] checkpoint_unverified_tags_ignored run_id=01M51WYGHQ6903KHS5WPAS25AC kind=last count=3 has_key=True
... (3 more log lines)
exit code: 1
-> recorded: matches the expectation


**What this shows.** Only the right key verifies the files as `authenticated` (exit 0). With another key, `verify` exits 1 and loads nothing. With no key at all, asking for `authenticated` is a configuration error (exit 2, the documented code for that), again without loading anything: it never quietly falls back to trusting the files. The reason shown for another key is `key_changed` (the run was signed with a different key); I had first expected `auth_invalid`, which the docs give for tags that fail under a *new* key after a rotation, but the section on a different key says `key_changed`, which is what happens. (The keys are random and never appear in the output.)

In [32]:
with timed("authenticated job killed"):
    seen, _ = watch(
        start_training("ck-auth2", env={"CVPILOT_CKPT_KEY": KEY}), at(3, 5), signal.SIGKILL
    )
E = {"CVPILOT_CKPT_KEY": KEY}
for label, env in (("no key", {}), ("another key", {"CVPILOT_CKPT_KEY": OTHER_KEY})):
    r = py(
        """
from cvpilot.ckpt import find_resume_point
from cvpilot.core.errors import CheckpointError

try:
    find_resume_point("ck-auth2")
    print("a resume point was returned (unexpected)")
except CheckpointError as exc:
    print("refused:", getattr(exc, "reason", None) or type(exc).__name__)
""",
        "key_restart_" + label.replace(" ", "_"),
        env=env,
        log_lines=0,
    )
    expected = "key_required" if label == "no key" else "key_changed"
    record(f"Restart of an authenticated job with {label}", 0, f"refused: {expected}", r)

[authenticated job killed: 3.8 s]


$ python key_restart_no_key.py
refused: key_required
... (4 more log lines)
exit code: 0
-> recorded: matches the expectation


$ python key_restart_another_key.py
refused: key_changed
... (4 more log lines)
exit code: 0
-> recorded: matches the expectation


**What this shows.** A job that was signed with a key cannot be resumed without that key (`key_required`) or with a different one (`key_changed`), even though the restart script asked for no particular trust level: once a run has been authenticated it never goes back to a weaker level on its own.

### Case 19. What could go wrong: saving at a step that already exists, with different bytes

A bug that reuses a step number, or a second process racing the first, would overwrite a good checkpoint with a different one.

In [33]:
r = py(
    """
import numpy as np

import cvpilot
from cvpilot.ckpt import CheckpointManager
from cvpilot.core.errors import CheckpointError

run = cvpilot.init("ck-same-step", task="classification")
ckpt = CheckpointManager(run, every_n_steps=1, install_signal_handlers=False)
first = np.full(4, 1.0, dtype=np.float32)
ckpt.save_now(1, model={"w": first})
print("saved step 1")
ckpt.save_now(1, model={"w": first})
print("saved step 1 again with identical bytes: accepted, nothing rewritten")
try:
    ckpt.save_now(1, model={"w": first + 1})
    print("different bytes at step 1 were saved (unexpected)")
except CheckpointError as exc:
    print("different bytes at step 1:", type(exc).__name__)
print("the original is still what a restore returns:", ckpt.restore().state["model"]["w"].tolist())
ckpt.close()
run.finish()
""",
    "same_step",
    log_lines=0,
)
record(
    "Save at an existing step with other bytes",
    0,
    "different bytes at step 1: CheckpointExistsError",
    r,
)

$ python same_step.py
saved step 1
saved step 1 again with identical bytes: accepted, nothing rewritten
different bytes at step 1: CheckpointExistsError
the original is still what a restore returns: [1.0, 1.0, 1.0, 1.0]
... (8 more log lines)
exit code: 0
-> recorded: matches the expectation


**What this shows.** The first checkpoint stays untouched. Saving the *same* bytes again is harmless and accepted; saving *different* bytes at a step that exists is refused with `CheckpointExistsError`. A checkpoint is never replaced, only added to.

### Case 20. What could go wrong: the disk is full

We cannot fill your disk, so the child process gets a small file-size limit instead (the operating system then refuses a big write exactly like a full disk would).

In [34]:
r = py(
    """
import resource
import signal

import numpy as np

import cvpilot
from cvpilot.ckpt import CheckpointManager
from cvpilot.core.errors import CheckpointError

run = cvpilot.init("ck-full-disk", task="classification")
ckpt = CheckpointManager(run, every_n_steps=1, keep_last=10, install_signal_handlers=False)
ckpt.save_now(1, model={"w": np.full(1000, 1.0, dtype=np.float32)})
print("saved step 1")
signal.signal(signal.SIGXFSZ, signal.SIG_IGN)  # report the refused write as an error instead of dying
resource.setrlimit(resource.RLIMIT_FSIZE, (1_000_000, 1_000_000))  # no file may grow beyond 1 MB
try:
    ckpt.save_now(2, model={"w": np.full(2_000_000, 2.0, dtype=np.float32)})  # an 8 MB state
    print("saved step 2 (unexpected)")
except CheckpointError as exc:
    print("step 2 could not be written:", type(exc).__name__)
ckpt.save_now(3, model={"w": np.full(1000, 3.0, dtype=np.float32)})
print("a later, smaller save works: step 3 saved")
print("restore returns step", ckpt.restore().manifest.step)
ckpt.close()
run.finish()
""",
    "full_disk",
    log_lines=0,
)
record("Disk full while saving a checkpoint", 0, "restore returns step 3", r)
record(
    "Disk full: the failed save raises an error to the training loop",
    0,
    "step 2 could not be written: CheckpointError",
    r,
)

$ python full_disk.py
saved step 1
step 2 could not be written: CheckpointError
a later, smaller save works: step 3 saved
restore returns step 3
... (8 more log lines)
exit code: 0
-> recorded: matches the expectation
-> recorded: matches the expectation


**What this shows.** The failed save raises a `CheckpointError` to your training loop (you can decide what to do) but it does not leave a half-written file behind: the earlier checkpoint is intact, and once there is room again the next save works and a restore returns the newest good state.

### Case 21. What could go wrong: a resumed run crashes too

Crash number one, a restart, crash number two. The second restart must continue from the *second* job's progress, not from the first one's old checkpoint.

In [35]:
with timed("three jobs: kill, kill, finish"):
    seen1, _ = watch(start_training("ck-twice", pause_ms=20), at(3, 30), signal.SIGKILL)
    seen2, _ = watch(start_training("ck-twice", pause_ms=20), at(5, 10), signal.SIGKILL)
    print("first job died in", seen1[-1].split(" loss")[0])
    print(
        "second job:",
        next(line for line in seen2 if line.startswith("resumed")),
        "... died in",
        seen2[-1].split(" loss")[0],
    )
    third = sh(
        "python train_clf.py --project ck-twice --epochs 6 --checkpoints",
        keep=True,
        lines=3,
        log_lines=0,
    )
sh("cvpilot runs list --project ck-twice", log_lines=0)
record(
    "Third start after two crashes continues from the second job's checkpoint",
    0,
    "resumed from step 380",
    third,
)

first job died in epoch 3/6 batch 30/95
second job: resumed from step 190 (epoch 3, batch 0) ... died in epoch 5/6 batch 10/95


$ python train_clf.py --project ck-twice --epochs 6 --checkpoints
resumed from step 380 (epoch 5, batch 0)
epoch 5/6 done: loss 0.017 val_accuracy 0.31
epoch 6/6 done: loss 0.007 val_accuracy 0.46
... (1 more lines)
... (10 more log lines)
exit code: 0
[three jobs: kill, kill, finish: 22.7 s]


$ cvpilot runs list --project ck-twice
RUN_ID                      STATUS     STARTED                    NAME
01M51WZEAEZDHT3K5W79ZXVW6G  finished   2026-10-16T08:24:09.468718+00:00  python-2026-10-16
01M51WZ5DY1DP6R14H2RCEYC2F  failed     2026-10-16T08:24:00.378671+00:00  python-2026-10-16
01M51WYVZ1HQAE4TE7M2Z828ZB  failed     2026-10-16T08:23:50.680107+00:00  python-2026-10-16
exit code: 0
-> recorded: matches the expectation


**What this shows.** The second job resumed from the end of epoch 2 (step 190) and got through epochs 3 and 4 before it was killed in epoch 5. The third job resumes from step 380, the end of epoch 4 and the newest checkpoint of the whole chain of runs, so the work of the second job was not lost. (An earlier version saved once right after every resume, which shifted the schedule by one step; the schedule is now seeded from the restored checkpoint.) The runs form a chain: each new run records the previous one as its parent.

### Case 22. What could go wrong: two jobs in the same project

Job A is still running (a live process). Job B starts in the same project and asks for a resume point. Taking A's checkpoint and marking A as failed would be a disaster: two processes would train the same thing.

In [36]:
with timed("two jobs in one project"):
    job_a = start_training("ck-two", pause_ms=60)
    seen_a = []
    for line in job_a.stdout:
        seen_a.append(line.rstrip())
        if line.startswith("epoch 1/6 done"):
            break
    print("job A:", seen_a[-1], "(still running)")
    job_b = sh(
        "python train_clf.py --project ck-two --epochs 1 --checkpoints",
        keep=True,
        lines=3,
        log_lines=0,
    )
    runs_now = sh("cvpilot runs list --project ck-two --json", quiet=True).json()
    print("statuses:", sorted(r["status"] for r in runs_now))
    job_a.kill()
    job_a.wait()
record("A live run is never recovered or resumed by a second job", 0, "starting fresh", job_b)
if "running" not in {r["status"] for r in runs_now}:
    raise RuntimeError("job A should still be running")

job A: epoch 1/6 done: loss 0.128 val_accuracy 0.35 (still running)


$ python train_clf.py --project ck-two --epochs 1 --checkpoints
starting fresh: no checkpoint to resume from
epoch 1/1 done: loss 0.128 val_accuracy 0.35
run 01M51WZV2B3S0BRQEWBRFMB18J finished
... (6 more log lines)
exit code: 0


statuses: ['finished', 'running']
[two jobs in one project: 12.1 s]
-> recorded: matches the expectation


**What this shows.** Job B printed `starting fresh`: it did not take over A's checkpoint, and A is still `running`, not marked failed. CV-Pilot decides that a run is alive from a lock that its process holds on its checkpoint folder (the operating system drops the lock the moment the process dies, so there is no timeout to guess). Tip: give separate jobs of a project different names with `cvpilot.init(name=...)` and pass `name=` to `find_resume_point` so they can never pick up each other's weights.

# The result table: expected against actual

In [37]:
width = [max(len(c[k]) for c in CASES) for k in ("case", "expected", "actual")]
width = [min(w, lim) for w, lim in zip(width, (62, 52, 70), strict=True)]


def cell(text, n):
    return (text if len(text) <= n else text[: n - 1] + "~").ljust(n)


print(
    f"{'#':>2}  {cell('case', width[0])}  {cell('expected (from the docs)', width[1])}  {cell('actual (this run)', width[2])}  match"
)
for n, c in enumerate(CASES, 1):
    print(
        f"{n:>2}  {cell(c['case'], width[0])}  {cell(c['expected'], width[1])}  {cell(c['actual'], width[2])}  {'yes' if c['ok'] else 'MISMATCH'}"
    )
mismatches = [c for c in CASES if not c["ok"]]
print(f"\nrows: {len(CASES)}, mismatches: {len(mismatches)}")

 #  case                                                            expected (from the docs)                              actual (this run)                                                       match
 1  Prediction scores contain NaN                                   exit 1, mentions 'score'                              exit 1, error: predictions have non-finite scores                       yes
 2  Predicted class id outside the class list                       exit 1, mentions 'outside [0, 80)'                    exit 1, error: predictions has class id 99 outside [0, 80)              yes
 3  Predictions for an image that is not in the ground truth        exit 1, mentions 'not in the evaluation set'          exit 1, error: predictions refers to image id <19 characters, sha256:~  yes
 4  Image without objects, declared as an empty row                 exit 0, mentions 'n_images: 128'                      exit 0, n_images: 128                                                   yes
 5  Imag

**What this shows.** Every row compares the exit code and a key phrase that was written down before the command ran with what the command really printed. A row that does not match is not edited: it is investigated, because it may be a real bug. Three rows were corrected while writing this notebook because the first guess was wrong, and three behaviours were fixed in CV-Pilot after the notebook exposed them (the notes next to cases 12, 16 and 21 say which).

### What you learned

* Bad numbers (`NaN`, impossible class ids, unknown images) stop at `cvpilot eval`; a model that finds nothing is valid, a missing file is a usage error.
* The gate fails closed: too few images, a metric or slice that cannot be checked, a different evaluation set, a run compared with itself, an unfinished baseline and an edited result all exit 1.
* An average can hide a loss: use per-class and per-slice limits next to the headline limit.
* Checkpoints survive truncation, deletion, planted files, wrong keys, a full disk, a second crash and a second job; every refusal is an explicit error with a reason, and the way through (`--accept-rollback-to`, `quarantine --clear`) is explicit and logged.